In [ ]:
# INSTALLAZIONE DIPENDENZE 
!pip install -q -U bitsandbytes
!pip install -q -U accelerate
!pip install -q -U datasets
!pip install -q -U git+https://github.com/huggingface/transformers.git

print("Installazione completata.")

In [ ]:
import pandas as pd
import json
import torch
import os
from tqdm import tqdm
from transformers import pipeline

# ==========================================
# 1. CONFIGURAZIONE LISTA DATASET
# ==========================================

BASE_KAGGLE_PATH = "/kaggle/input/datasets/silvioliparoti/dpo-dataset-dpo-v-2" 

DATASETS = [
    {
        "input": os.path.join(BASE_KAGGLE_PATH, "dataset_LPR_new_DPO_v2_1.csv"),
        "output": "dpo_dataset_ready_v2_1.jsonl"
    },
    {
        "input": os.path.join(BASE_KAGGLE_PATH, "dataset_LPR_new_DPO_v2_2.csv"),
        "output": "dpo_dataset_ready_v2_2.jsonl"
    }
]

MODEL_ID = "HuggingFaceH4/zephyr-7b-beta" 

# ==========================================
# 2. CARICAMENTO MODELLO TEACHER (UNA SOLA VOLTA)
# ==========================================
print(f"Carico il modello Teacher: {MODEL_ID}...")

# torch_dtype=torch.bfloat16 risparmia tantissima memoria sulla GPU
pipe = pipeline(
    "text-generation", 
    model=MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

system_prompt = "You are a literal, boring assistant. You ruin jokes by explaining them literally or rewriting them as serious statements."

# ==========================================
# 3. ELABORAZIONE MULTIPLA (IL TUO LOOP)
# ==========================================
for config in DATASETS:
    input_file = config["input"]
    output_file = config["output"]
    
    print("\n" + "="*60)
    print(f"LAVORAZIONE DATASET: {input_file}")
    print("="*60)
    
    # --- A. Caricamento e Ordinamento ---
    if not os.path.exists(input_file):
        print(f" ERRORE: File non trovato {input_file}")
        continue
        
    df = pd.read_csv(input_file)
    
    # MAGIA SOTA: Ordiniamo in base alla nuova Ground Truth e prendiamo l'Elite
    df = df.sort_values(by='LPR_new', ascending=False)
    df_subset = df.head(2000).copy()
    
    text_col = "jokeText" if "jokeText" in df_subset.columns else "text"
    jokes = df_subset[text_col].tolist()
    print(f" Caricate {len(jokes)} battute Gold (Elite Top 2000).")
    
    # --- B. Generazione Coppie ---
    dpo_data = []
    print(f"Inizio generazione versioni noiose per {output_file}...")
    
    for joke in tqdm(jokes):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Rewrite the following joke into a completely serious, boring, and literal sentence. Remove all humor. Keep it short.\n\nJoke: \"{joke}\""}
        ]
        
        try:
            outputs = pipe(
                messages, 
                max_new_tokens=60,   
                do_sample=True,      
                temperature=0.7,     
                top_k=50,            
                top_p=0.95           
            )
            
            generated = outputs[0]["generated_text"][-1]["content"]
            boring_text = generated.replace('"', '').replace("Here is a serious version:", "").strip()

            entry = {
                "prompt": "Tell me a joke.", 
                "chosen": joke,               
                "rejected": boring_text       
            }
            dpo_data.append(entry)
            
        except Exception as e:
            print(f" Errore su battuta: {e}")
            continue

    # --- C. Salvataggio ---
    with open(output_file, 'w') as f:
        for entry in dpo_data:
            json.dump(entry, f)
            f.write('\n')

    print(f" FATTO! Dataset salvato: {output_file}")
    print(f" Contiene {len(dpo_data)} coppie pronte.")

print("\n TUTTE LE ELABORAZIONI COMPLETATE!")